# Session 2 — Object Selection and Event Reconstruction

**Learning objectives:**
- Understand physics object selection in CMS
- Apply jet quality and kinematic cuts
- Use b-tagging (DeepJet discriminator)
- Implement a lepton veto
- Build event-level selection logic

**Duration:** ~3 hours

---
## 1. Physics Objects in CMS

Reconstructed objects (jets, electrons, muons, MET) are built from detector signals. We apply **identification** and **selection** criteria to reduce fake rates and align with analysis definitions.

For the bbDM search we need:
- **Good jets** (quality + kinematic cuts)
- **b-tagged jets** (heavy-flavor tagging)
- **No tight leptons** (lepton veto)

---
## 2. Jet Selection Criteria

Typical jet selection for Run-2 analyses:
- **pT > 30 GeV** (above noise and trigger thresholds)
- **|η| < 2.4** (tracker coverage for b-tagging)
- **Jet ID** (quality flags; in NanoAOD often `jetId >= 2` for tight)

We will implement this in Coffea using a **mask** on the jet array.

In [ ]:
import numpy as np
import awkward as ak

JET_PT_MIN = 30.0   # GeV
JET_ETA_MAX = 2.4

def select_good_jets(events):
    jets = events.Jet
    mask = (
        (jets.pt > JET_PT_MIN) &
        (np.abs(jets.eta) < JET_ETA_MAX) &
        (jets.jetId >= 2)
    )
    return jets[mask]

print("Function select_good_jets defined. Call it with your events.")

### Exercise 2.1
Load your events (as in Session 1) and apply `select_good_jets`. Compare the number of jets **before** and **after** selection (e.g. total jet count and jet multiplicity per event).

In [ ]:
# Your code: load events, get good_jets, compare multiplicities
# good_jets = select_good_jets(events)
# njets_before = ak.num(events.Jet)
# njets_after = ak.num(good_jets)
# print("Mean jets/event before:", ak.mean(njets_before))
# print("Mean jets/event after:", ak.mean(njets_after))

---
## 3. Jet Quality Flags

The **jetId** in NanoAOD encodes quality:
- Loose (1), Tight (2), TightLeptonVeto (4).
- We use `jetId >= 2` for tight jets.

This removes jets from noise and problematic regions.

---
## 4. Introduction to b-Tagging

**b-tagging** identifies jets likely from b quarks using secondary vertices and track impact parameters. CMS provides discriminants from multivariate algorithms.

We use **DeepJet** (or DeepFlav) and the branch **btagDeepFlavB**. A working point (WP) is a threshold on this discriminant; higher threshold = purer b-jets, lower efficiency.

---
## 5. DeepJet Discriminator

**Medium working point** (example for 2017):
`btagDeepFlavB > 0.2783`

Count b-jets per event by summing jets that pass this cut.

In [ ]:
BTAG_WP_MEDIUM = 0.2783  # DeepFlavB medium (2017); adjust for your NanoAOD year

def count_bjets(jets, wp=BTAG_WP_MEDIUM):
    return ak.sum(jets.btagDeepFlavB > wp, axis=1)

# After getting good_jets:
# nbjets = count_bjets(good_jets)
# print("Example b-jet counts (first 10 events):", nbjets[:10])

### Exercise 2.2
Using `good_jets` and `count_bjets`, plot the **b-jet multiplicity** (0, 1, 2, 3, ...). Use a histogram with bins 0–6. Label axes.

In [ ]:
import matplotlib.pyplot as plt

# Your code: b-jet multiplicity histogram
# nbjets = count_bjets(good_jets)
# plt.hist(ak.to_numpy(nbjets), bins=range(7), ...)

---
## 6. Lepton Identification

For the **lepton veto** we need to define **tight** electrons and muons. Events with one or more tight leptons are rejected.

Typical requirements:
- **Electrons:** pT > 10 GeV, |η| < 2.5, cutBased ID (e.g. tight = 4 in older NanoAOD, or >= 2)
- **Muons:** pT > 10 GeV, |η| < 2.4, tightId, isolation (e.g. pfRelIso04_all < 0.15)

In [ ]:
LEP_PT_MIN = 10.0
LEP_ETA_MAX_EL = 2.5
LEP_ETA_MAX_MU = 2.4

def select_tight_electrons(events):
    ele = events.Electron
    mask = (
        (ele.pt > LEP_PT_MIN) &
        (np.abs(ele.eta) < LEP_ETA_MAX_EL) &
        (ele.cutBased >= 2)
    )
    return ele[mask]

def select_tight_muons(events):
    mu = events.Muon
    mask = (
        (mu.pt > LEP_PT_MIN) &
        (np.abs(mu.eta) < LEP_ETA_MAX_MU) &
        (mu.tightId) &
        (mu.pfRelIso04_all < 0.15)
    )
    return mu[mask]

def n_tight_leptons(events):
    nele = ak.count(select_tight_electrons(events).pt, axis=1)
    nmu = ak.count(select_tight_muons(events).pt, axis=1)
    return nele + nmu

print("Lepton selection functions defined.")

### Exercise 2.3
Compute `n_tight_leptons(events)` and plot the distribution (0, 1, 2, ...). How many events have exactly 0 tight leptons?

In [ ]:
# Your code: n_tight_leptons and plot
# nlep = n_tight_leptons(events)
# plt.hist(ak.to_numpy(nlep), bins=range(6), ...)
# print("Events with 0 tight leptons:", ak.sum(nlep == 0))

---
## 7. Event Cleaning

In full analyses we also apply **event filters** (e.g. bad muons, duplicate events). For this exercise we keep the selection as: good jets, b-tag count, and lepton veto.

**Event selection so far:**
- At least one good jet (for plots)
- Optionally: ≥2 b-jets and 0 leptons for a pre-signal region

---
## 8. Putting It Together

Build a mask for events that pass:
- ≥1 good jet
- (Optional) ≥2 b-jets, 0 tight leptons

Then plot **jet multiplicity**, **b-jet multiplicity**, and **leading jet pT** after this selection.

In [ ]:
# Example: event mask with good jets only
# good_jets = select_good_jets(events)
# njets = ak.num(good_jets)
# nbjets = count_bjets(good_jets)
# nlep = n_tight_leptons(events)
# mask = (njets >= 1)
# good_events = events[mask]
# good_jets_sel = good_jets[mask]
# leading_pt = ak.fill_none(ak.firsts(good_jets_sel.pt), 0.0)

### Exercise 2.4 — Jet multiplicity after selection
Plot **jet multiplicity** (using `ak.num(good_jets)`) for events with at least one good jet. Use bins 0–15.

In [ ]:
# Your code: jet multiplicity after selection

### Exercise 2.5 — b-jet multiplicity
Plot **b-jet multiplicity** (same as Exercise 2.2; you can overlay or plot separately).

In [ ]:
# Your code: b-jet multiplicity

### Exercise 2.6 — Leading jet pT
Plot the **leading (highest pT) jet pT** per event. Use `ak.firsts(good_jets.pt)` (with `ak.fill_none(..., 0)` if needed). Range 0–500 GeV.

In [ ]:
# Your code: leading jet pT
# lead_pt = ak.fill_none(ak.firsts(good_jets.pt), 0.0)
# plt.hist(ak.to_numpy(lead_pt), bins=50, range=(0, 500), ...)

---
## 9. Modify Thresholds (Discussion)

**Try changing:**
- Jet pT cut from 30 to 40 GeV. How do jet and b-jet multiplicities change?
- b-tag WP: use a looser value (e.g. 0.2) or tighter (e.g. 0.5). How does b-jet multiplicity change?

Discuss with your neighbour: why does tt̄ dominate when we require ≥2 b-jets?

In [ ]:
# Optional: re-run with JET_PT_MIN = 40 and compare histograms

---
## 10. Summary — Session 2

- **Good jets:** pT > 30 GeV, |η| < 2.4, jetId >= 2.
- **b-jets:** btagDeepFlavB > 0.2783 (medium WP).
- **Lepton veto:** 0 tight electrons and 0 tight muons.
- We built event-level quantities: njets, nbjets, nlep, leading jet pT.

**Next session:** We will define the signal region (MET > 200 GeV, ≥2 b-jets, 0 leptons), compare yields, and introduce control regions.